# Аналіз витрат на відрядження

Цей ноутбук демонструє, як створювати агентів, які використовують плагіни для обробки витрат на відрядження з локальних фотографій квитанцій, створення електронного листа з претензією на відшкодування витрат і візуалізації даних про витрати за допомогою кругової діаграми. Агенти динамічно обирають функції залежно від контексту завдання.

Кроки:
1. Агент OCR обробляє локальне зображення квитанції та витягує дані про витрати на відрядження.
2. Агент електронної пошти генерує електронний лист із претензією на відшкодування витрат.

### Приклад сценарію витрат на відрядження:
Уявіть, що ви співробітник, який подорожує для ділової зустрічі в іншому місті. У вашої компанії є політика відшкодування всіх обґрунтованих витрат, пов’язаних із поїздкою. Ось розподіл потенційних витрат на відрядження:
- Транспорт:
Авіаквитки на кругову поїздку з вашого рідного міста до міста призначення.
Таксі або служби по виклику автомобілів до і від аеропорту.
Місцевий транспорт у місті призначення (наприклад, громадський транспорт, оренда авто або таксі).

- Проживання:
Перебування в готелі на три ночі в бізнес-готелі середнього рівня, близько до місця зустрічі.

- Харчування:
Щоденна добова норма на сніданок, обід і вечерю відповідно до правил компанії.

- Інші витрати:
Плата за паркування в аеропорту.
Плата за доступ до інтернету в готелі.
Чаєві або невеликі сервісні збори.

- Документація:
Ви подаєте всі квитанції (авіаквитки, таксі, готель, харчування тощо) і заповнену форму звіту про витрати для відшкодування.


## Імпортуйте необхідні бібліотеки

Імпортуйте необхідні бібліотеки та модулі для ноутбука.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Визначення моделей витрат

Створіть модель Pydantic для окремих витрат і клас ExpenseFormatter для перетворення запиту користувача у структуровані дані про витрати.

Кожна витрата буде представлена у форматі:
`{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Визначення інструментів - Генерація електронної пошти

Створіть функцію інструменту для генерації електронної пошти для подання заяви на відшкодування витрат.
- Цей інструмент використовує декоратор `@tool` з Microsoft Agent Framework.
- Він обчислює загальну суму витрат і форматує деталі у тіло електронної пошти.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Інструмент для вилучення витрат на подорожі з зображень чеків

Створіть функцію інструменту для вилучення витрат на подорожі з зображень чеків.
- Цей інструмент використовує декоратор `@tool` з Microsoft Agent Framework.
- Він читає зображення чека, кодує його у base64 та повертає data URI для аналізу агентом.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Обробка витрат

Визначте агентів і з'єднайте їх у послідовний робочий процес за допомогою `WorkflowBuilder`.
- Агент OCR витягує структуровані дані про витрати з зображення квитанції, використовуючи інструмент `load_receipt_image`.
- Агент Email бере витягнуті дані і генерує професійний лист із претензією на відшкодування витрат за допомогою інструменту `generate_expense_email`.
- `WorkflowBuilder` з `add_edge` створює послідовний конвеєр: агент OCR → агент Email.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Головна функція

Побудуйте послідовний робочий процес і запустіть його для обробки зображення чека та створення електронного листа для заяви на відшкодування витрат.


> **Примітка:** Цей робочий процес наразі передає зображення чека у вигляді тексту base64, який більшість чат-моделей (включаючи gpt-4o) не розглядатимуть як зображення.  
> Це також може перевищувати контекстне вікно моделі. Рекомендується запускати OCR за допомогою Azure AI Vision (або іншого інструменту OCR) і передавати тільки вилучений текст, або змінити процес, щоб надсилати зображення як повідомлення з `image_url`.  
> Якщо ви лише хочете уникнути помилок контексту, спробуйте використовувати менше зображення чека або модель з більшим контекстним вікном.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
